# Simulating Future Waves in Sri Lanka
Based on the wave climate projection, it is concluded that the future storm intensity is decreasing. Three cases will be investigated 
Get shoreline of 1% exceedance with: 
1. current wave climate, no SLR 
2. current wave climate, with SLR 
3. future wave climate, with SLR 

In [ ]:
# check how the lambda is built
import os

import pandas as pd
import numpy as np 

import plotly.express as px 
import matplotlib.pyplot as plt

from pcr.model import PCRModel

In [ ]:
# Configure path 
DIR_DATA = '../data'
DIR_OUT = os.path.join(DIR_DATA, 'output')
SLR_PATH = os.path.join(DIR_DATA, 'AR6_slr')
ERA5_PATH = os.path.join(DIR_DATA, 'ERA5/B3_offshore.nc')

# initialize simulation length and number of sim
year_start = '2020'
year_end = '2119'
nr_sim = 100000
nr_bat = 1000

# no change in sl and storm, current sea level, current wave (clcw)
clcw = PCRModel(
    year_start=year_start,
    year_end=year_end,
    nr_simulation=nr_sim,
    nr_batch=nr_bat,
    # SLR 
    scenario='0',
    # wave data
    wave_data_path=ERA5_PATH,
    # future condition 
    fac_lambda=np.array([[2015, 2100], [1, 1]])
)

# SSP5-8.5, SLR only. future sea level, current wave (flcw)
flcw = PCRModel(
    year_start=year_start,
    year_end=year_end,
    nr_simulation=nr_sim,
    nr_batch=nr_bat,
    # SLR 
    scenario='ssp585',
    slr_data_path=SLR_PATH,
    # wave data
    wave_data_path=ERA5_PATH,
    # future condition 
    fac_lambda=np.array([[2015, 2100], [1, 1]])
)

# SSP5-8.5, SLR and Storm Intensity . Future sea level, future wave (FLFW)
flfw = PCRModel(
    year_start=year_start,
    year_end=year_end,
    nr_simulation=nr_sim,
    nr_batch=nr_bat,
    # SLR 
    scenario='ssp585',
    slr_data_path=SLR_PATH,
    # wave data
    wave_data_path=ERA5_PATH,
    # future condition 
    fac_lambda=np.array([[2015, 2100], [1, 0.9]])
)

flfw_inc = PCRModel(
    year_start=year_start,
    year_end=year_end,
    nr_simulation=nr_sim,
    nr_batch=nr_bat,
    # SLR 
    scenario='ssp585',
    slr_data_path=SLR_PATH,
    # wave data
    wave_data_path=ERA5_PATH,
    # future condition 
    fac_lambda=np.array([[2015, 2100], [1, 1.1]])
)

In [ ]:
# run simulation 
flfw_inc.run()

In [ ]:
np.savetxt(os.path.join(DIR_OUT, 'flfw_annualmaxima.csv'), flfw.shoreline_stats, delimiter=',')

In [ ]:
def calculate_finalshoreline(model, n_year=3):
    t_th = model.t_days - 365.25 * n_year

    return np.asarray([
        s[np.searchsorted(t, t_th, side='right'):].mean()
        for t, s in zip(model.track_time, model.track_shoreline)
    ])

In [ ]:
flfw_inc.final_shoreline = calculate_finalshoreline(flfw_inc)
np.savetxt(os.path.join('flfw_finalshoreline.csv'), flfw.final_shoreline, delimiter=',')

# Visualize the results 

In [ ]:
viz_maxima_flfwi = flfw_inc.shoreline_stats
viz_final_flfwi = flfw_inc.final_shoreline
del flfw_inc

viz_maxima_clcw = pd.read_csv(os.path.join(DIR_OUT, 'clcw_annualmaxima.csv'), header=None).to_numpy()
viz_final_clcw = pd.read_csv(os.path.join(DIR_OUT, 'clcw_finalshoreline.csv'), header=None).to_numpy()

viz_maxima_flcw = pd.read_csv(os.path.join(DIR_OUT, 'flcw_annualmaxima.csv'), header=None).to_numpy()
viz_final_flcw = pd.read_csv(os.path.join(DIR_OUT, 'flcw_finalshoreline.csv'), header=None).to_numpy()

viz_maxima_flfw = pd.read_csv(os.path.join(DIR_OUT, 'flfw_annualmaxima.csv'), header=None).to_numpy()
viz_final_flfw = pd.read_csv(os.path.join(DIR_OUT, 'flfw_finalshoreline.csv'), header=None).to_numpy()

In [ ]:
# exceedance probability plot 
years = [25, 50, 80, 100]
colors = ['k', 'k', 'r', 'r']
dashes = ['-', '--', '-', '--']

maximas = [viz_maxima_clcw, viz_maxima_flcw, viz_maxima_flfw, viz_maxima_flfwi]
finals = [viz_final_clcw, viz_final_flcw, viz_final_flfw, viz_final_flfwi.reshape((100000,1))]

case_title = ['Current Storm, No SLR', 'Current Storm, with SLR', 'Future Storm (-10%), with SLR', 'Future Storm (+10%), with SLR']

fig, ax = plt.subplots(1, 4, figsize=(12,4))
exceedance = np.linspace(0, 100, 100000) 

for i, v in enumerate(maximas):
    # select data 
    for year, color, dash in zip(years, colors, dashes):
        sorted_year = np.sort(-v[year - 1])[::-1]
        ax[i].plot(sorted_year, exceedance, label=str(year+2020), color=color, linestyle=dash)

    ax[i].set_yscale('log')
    ax[i].set_ylim([1, 100])
    ax[i].set_xlim([-50, 150])
    ax[i].set_xlabel('Recession (m)')
    ax[i].grid(True, which='both', alpha=0.3)
    ax[i].set_title(case_title[i])

# legend 
ax[2].legend(title='Year')

# y
ax[0].set_ylabel('Exceedance Probability (%)')

fig.suptitle('Exceedance Plot for SSP5-8.5 Scenario Projection')
plt.tight_layout()
plt.show()


In [ ]:
for case, color, title in zip(maximas, colors, case_title):
    print(f'{color}, {title}')

In [ ]:
# exceedance probability plot 
years = [25, 50, 100]
colors = ['k', 'r', 'g', 'g']
dashes = ['-', '-', '-', '--']

fig, ax = plt.subplots(1, 3, figsize=(10,4))
exceedance = np.linspace(0, 100, 100000) 

for i, y in enumerate(years): 
    # plot lines for every case 
    for case, color, title, dash in zip(maximas, colors, case_title, dashes):
        sorted_year = np.sort(-case[y-1])[::-1]
        ax[i].plot(sorted_year, exceedance, label=title, color=color, linestyle=dash)

    ax[i].set_yscale('log')
    ax[i].set_ylim([1, 100])
    ax[i].set_xlim([-50, 150])
    ax[i].set_xlabel(f'Recession by {y+2020} (m)')
    ax[0].set_ylabel('Exceedance Probability (%)')

    ax[i].grid(True, which='both', alpha=0.3)
    ax[i].set_title(f'{y+2020}')

# legend 
# ax[2].legend(title='Case')

# y

# fig.suptitle('Exceedance Plot for every year')
handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, 
           bbox_to_anchor=(0.5, -0.05), frameon=False)

plt.tight_layout()
plt.show()


In [ ]:
years_list = np.arange(1,100)

results = {name: [] for name in case_title}
results2 = {name: [] for name in case_title}
results3 = {name: [] for name in case_title}

for i, v in enumerate(maximas):  # i = case index
    for year in years_list:
        data = -v[year - 1]
        
        # sort descending (largest recession first)
        sorted_vals = np.sort(data)[::-1]
        n = len(sorted_vals)
        
        # rank-based exceedance probability (Weibull plotting position), in %
        ranks = np.arange(1, n + 1)
        exceed_prob = ranks / (n + 1) * 100
        
        # interpolate to find value at exactly 1% exceedance
        # np.interp needs x increasing, so reverse both arrays
        value_at_1pct = np.interp(1, exceed_prob, sorted_vals)
        value_at_10pct = np.interp(10, exceed_prob, sorted_vals)
        value_at_50pct = np.interp(50, exceed_prob, sorted_vals)
        
        results[case_title[i]].append(value_at_1pct)
        results2[case_title[i]].append(value_at_10pct)
        results3[case_title[i]].append(value_at_50pct)

# results now looks like: {'Current Storm, No SLR': [val_2045, val_2070, val_2095, val_2120], ...}

In [ ]:
actual_years = [2020 + y for y in years_list]  # e.g. [2045, 2070, 2100, 2120]

fig, ax = plt.subplots(1, 3, figsize=(12,5))
all_results = [results, results2, results3]
pct = ['1', '10', '50']

for i, r in enumerate(all_results):
    for name, color, dash in zip(case_title, colors, dashes):
        ax[i].plot(actual_years, r[name], color=color, label=name, linestyle=dash)

    ax[i].set_xlabel('Year')
    ax[i].set_ylabel(f'Recession at {pct[i]}% Exceedance Probability (m)')
    ax[i].set_title(f'{pct[i]}% Exceedance Recession vs Year')
    ax[i].set_ylim([0, 120])
    ax[i].grid(alpha=0.3)

handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, 
           bbox_to_anchor=(0.5, -0.05), frameon=False)
    
plt.tight_layout()
plt.show()

# Low Emission Scenario 

In [ ]:
# no change in sl and storm, current sea level, current wave (clcw)

# SSP1-2.6, SLR only. future sea level, current wave (flcw)
flcw_126 = PCRModel(
    year_start=year_start,
    year_end=year_end,
    nr_simulation=nr_sim,
    nr_batch=nr_bat,
    # SLR 
    scenario='ssp126',
    slr_data_path=SLR_PATH,
    # wave data
    wave_data_path=ERA5_PATH,
    # future condition 
    fac_lambda=np.array([[2015, 2100], [1, 1]])
)

# SSP1-2.6, SLR and Storm Intensity . Future sea level, future wave (FLFW)
flfw_126 = PCRModel(
    year_start=year_start,
    year_end=year_end,
    nr_simulation=nr_sim,
    nr_batch=nr_bat,
    # SLR 
    scenario='ssp126',
    slr_data_path=SLR_PATH,
    # wave data
    wave_data_path=ERA5_PATH,
    # future condition 
    fac_lambda=np.array([[2015, 2100], [1, 1.03]])
)

In [ ]:
maximas_126 = [viz_maxima_clcw]
finals_126 = [viz_final_clcw]

for m in [flcw_126, flfw_126]:
    m.run()

    maximas_126.append(m.shoreline_stats)
    m.final_shoreline = calculate_finalshoreline(m)
    finals_126.append(m.final_shoreline.reshape((100000,1)))

In [ ]:
CASE_FOLDER = os.path.join(DIR_OUT, 'SL_future_ssp126')
os.makedirs(CASE_FOLDER, exist_ok=True)

case_code = ['clcw', 'flcw', 'flfw_d', 'flfw']

for i, code in enumerate(case_code):
    filename = f'{code}_annualmaxima.csv'
    np.savetxt(os.path.join(CASE_FOLDER, filename), maximas_126[i], delimiter=',')
    filename = f'{code}_finalshoreline.csv'
    np.savetxt(os.path.join(CASE_FOLDER, filename), finals_126[i], delimiter=',')

In [ ]:
# plot Excedeence Probability of Final Coastline position in 2120
case_126 = ['Current Storm, No SLR', 'Current Storm, SLR SSP1-2.6', 'Future Storm (-10%), SLR SSP1-2.6', 'Future Storm (+3%), SLR SSP1-2.6']
fig, ax = plt.subplots(figsize=(8, 6))

for i, (v, c, d) in enumerate(zip(finals_126, colors, dashes)):
    data = -np.concat(v)

    # sort descending (largest recession first)
    sorted_vals = np.sort(data)[::-1]
    n = len(sorted_vals)

    # rank-based exceedance probability (Weibull plotting position), in %
    ranks = np.arange(1, n + 1)
    exceed_prob = ranks / (n + 1) * 100

    ax.plot(exceed_prob, sorted_vals, color=c, label=case_126[i], linestyle=d)

ax.set_xlabel('Exceedance probability (%)')
ax.set_ylabel('Recession by 2120 (m)')
ax.set_xscale('log')
ax.grid(True, which='both', alpha=0.3)
ax.set_xlim(50,1)
ax.set_ylim(-10,150)
ax.legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(1, 4, figsize=(12,4))
exceedance = np.linspace(0, 100, 100000) 

for i, v in enumerate(maximas_126):
    # select data 
    for year, color, dash in zip(years, colors, dashes):
        sorted_year = np.sort(-v[year - 1])[::-1]
        ax[i].plot(sorted_year, exceedance, label=str(year+2020), color=color, linestyle=dash)

    ax[i].set_yscale('log')
    ax[i].set_ylim([1, 100])
    ax[i].set_xlim([-50, 150])
    ax[i].set_xlabel('Recession (m)')
    ax[i].grid(True, which='both', alpha=0.3)
    ax[i].set_title(case_126[i])

# legend 
ax[2].legend(title='Year')

# y
ax[0].set_ylabel('Exceedance Probability (%)')

fig.suptitle('Exceedance Plot for SSP1-2.6 Scenario Projection')
plt.tight_layout()
plt.show()

# Changing Recovery Rate

In [ ]:
model = PCRModel(
    year_start='2020',
    year_end='2119',
    nr_simulation=100000,
    nr_batch=1000,
    # SLR 
    scenario='ssp585',
    slr_data_path=SLR_PATH,
    # wave data
    wave_data_path=ERA5_PATH,
    # future condition 
    fac_lambda=np.array([[2015, 2100], [1, 0.9]])
)

# run the rec_rate-independent stages once
model.init_slr()
model.load_wave_data()
model.detect_storms()

In [ ]:
# make the case: decreasing storm frequency, varying recovery rate
# decrease storm compare with clcw and flcw SSP5-8.5
model.scenario = 'ssp585'
model.fac_lambda = np.array([[2015, 2100], [1, 0.9]])
model.rec_rate_end = (6.6919 / 365) # based on the calibrated value to 

model.init_slr()
model.run_simulation()

In [ ]:
model.track_shoreline = None
model.track_time = None

In [ ]:
viz_maxima_clcw = pd.read_csv(os.path.join(DIR_OUT, 'SL_future_waves_10dec_ssp585', 'clcw_annualmaxima.csv'), header=None).to_numpy()
viz_final_clcw = pd.read_csv(os.path.join(DIR_OUT, 'SL_future_waves_10dec_ssp585', 'clcw_finalshoreline.csv'), header=None).to_numpy()

viz_maxima_flcw = pd.read_csv(os.path.join(DIR_OUT, 'SL_future_waves_10dec_ssp585', 'flcw_annualmaxima.csv'), header=None).to_numpy()
viz_final_flcw = pd.read_csv(os.path.join(DIR_OUT, 'SL_future_waves_10dec_ssp585', 'flcw_finalshoreline.csv'), header=None).to_numpy()

In [ ]:
maximas = [viz_maxima_clcw, viz_maxima_flcw, model.shoreline_stats]

In [ ]:
years_list = np.arange(1,100)

case_title = ['Current Storm, No SLR',
 'Current Storm, with SLR',
 'Future Storm (-10%), with SLR',]

results = {name: [] for name in case_title}
results2 = {name: [] for name in case_title}
results3 = {name: [] for name in case_title}

for i, v in enumerate(maximas):  # i = case index
    for year in years_list:
        data = -v[year - 1]
        
        # sort descending (largest recession first)
        sorted_vals = np.sort(data)[::-1]
        n = len(sorted_vals)
        
        # rank-based exceedance probability (Weibull plotting position), in %
        ranks = np.arange(1, n + 1)
        exceed_prob = ranks / (n + 1) * 100
        
        # interpolate to find value at exactly 1% exceedance
        # np.interp needs x increasing, so reverse both arrays
        value_at_1pct = np.interp(1, exceed_prob, sorted_vals)
        value_at_10pct = np.interp(10, exceed_prob, sorted_vals)
        value_at_50pct = np.interp(50, exceed_prob, sorted_vals)
        
        results[case_title[i]].append(value_at_1pct)
        results2[case_title[i]].append(value_at_10pct)
        results3[case_title[i]].append(value_at_50pct)

# results now looks like: {'Current Storm, No SLR': [val_2045, val_2070, val_2095, val_2120], ...}

In [ ]:
actual_years = [2020 + y for y in years_list]  # e.g. [2045, 2070, 2100, 2120]

fig, ax = plt.subplots(1, 3, figsize=(12,5))
all_results = [results, results2, results3]
pct = ['1', '10', '50']

for i, r in enumerate(all_results):
    for name, color in zip(case_title, ['k', 'r', 'g']):
        ax[i].plot(actual_years, r[name], color=color, label=name)

    ax[i].set_xlabel('Year')
    ax[i].set_ylabel(f'Recession at {pct[i]}% Exceedance Probability (m)')
    ax[i].set_title(f'{pct[i]}% Exceedance Recession vs Year')
    ax[i].set_ylim([0, 120])
    ax[i].grid(alpha=0.3)

handles, labels = ax[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='lower center', ncol=4, 
           bbox_to_anchor=(0.5, -0.05), frameon=False)
fig.suptitle("Varying recovery rate")

plt.tight_layout()
plt.show()

In [ ]:
CASE_FOLDER = os.path.join(DIR_OUT, 'SL_future_ssp585_varrecrate')
os.makedirs(CASE_FOLDER, exist_ok=True)

np.savetxt(os.path.join(CASE_FOLDER, 'flfw_annualmaxima.csv'), model.shoreline_stats, delimiter=',')